In [1]:
from pathlib import Path
import sys
import os

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

from import_libraries import *  # NOSONAR
from global_variables import *  # NOSONAR
import common_libraries.io_library as io_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\xicor\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
WORK_PATH = Path("C:/work_dir/kozani-2023-2024")
# PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
# OUTPUT_PATH = WORK_PATH / "3_ml_classification_oversample"

# PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_annual_stats.geoparquet"
# OUTPUT_PATH = WORK_PATH / "3_ml_classification_reduced"

# PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
# OUTPUT_PATH = WORK_PATH / "3_ml_classification_tune_probabilities"

PARCELS_PATH = WORK_PATH / "1_parcel_stats" / "satellite_parcel_time_stats.geoparquet"
OUTPUT_PATH = WORK_PATH / "3_ml_classification_tune_probabilities1"

class_column = "label"
parcel_id_column = "parcel_code"

In [3]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
gdf_parcels.head(3)

Reading file: C:/work_dir/kozani-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet


parcel_code  label             geometry  batch_number  eligible_pixel_count  \
0   266124977    110  POLYGON ((346567...             3                   44    
1    26857194    119  POLYGON ((336782...             3                   33    
2    26857195    119  POLYGON ((335690...             3                   89    

   meets_minimum_pixel_count  parcel_area_m2  approx_pixel_count  \
0                 True       4622.0831583770       46.2208315838   
1                 True       3296.4026005738       32.9640260057   
2                 True       8657.4707037060       86.5747070371   

   B02_bowley_skewness__20231001  B02_bowley_skewness__20231101  \
0        -0.2222223536                  -0.2606636878             
1         0.2777778640                   0.3008132642             
2         0.0789472523                   0.5600000745             

   B02_bowley_skewness__20231201  B02_bowley_skewness__20240101  \
0         0.3705885962                  -0.2437950298             
1         0.1538461355                   0.2293576163             
2         0.2142859519                   0.0392156147             

   B02_bowley_skewness__20240201  B02_bowley_skewness__20240301  \
0         0.0317449805                  -0.0531940864             
1         0.3023259239                  -0.1886796166             
2         0.2786888450                   0.6046514146             

   B02_bowley_skewness__20240401  B02_bowley_skewness__20240501  \
0         0.2727292431                   0.3005467017             
1         0.6962025615                   0.5952381903             
2         0.3614041600                   0.4193546837             

   B02_bowley_skewness__20240601  B02_bowley_skewness__20240701  \
0        -0.1412439525                   0.1864407636             
1         0.2212392881                  -0.3518521585             
2         0.0000000000                   0.2336454455             

   B02_bowley_skewness__20240801  B02_bowley_skewness__20240901  \
0         0.1384614679                  -0.0684901867             
1        -0.0140842262                  -0.2920352465             
2         0.1290320255                   0.2677169835             

   B02_coefficient_of_variation__20231001  \
0         0.1539387617                      
1         0.1977170996                      
2         0.2083519887                      

   B02_coefficient_of_variation__20231101  \
0         0.1340517814                      
1         0.1226033857                      
2         0.2390885263                      

   B02_coefficient_of_variation__20231201  \
0         0.1555584573                      
1         0.2268157149                      
2         0.2487263917                      

   B02_coefficient_of_variation__20240101  \
0         0.1321227188                      
1         0.2087659311                      
2         0.1694184639                      

   B02_coefficient_of_variation__20240201  \
0         0.0979855404                      
1         0.1757070871                      
2         0.1864479979                      

   B02_coefficient_of_variation__20240301  \
0         0.1214867041                      
1         0.3014574891                      
2         0.2711784960                      

   B02_coefficient_of_variation__20240401  \
0         0.1451981675                      
1         0.2926339985                      
2         0.2288398794                      

   B02_coefficient_of_variation__20240501  \
0         0.2022535334                      
1         0.2349692473                      
2         0.1295184390                      

   B02_coefficient_of_variation__20240601  \
0         0.1720276301                      
1         0.1197639817                      
2         0.1128968037                      

   B02_coefficient_of_variation__20240701  \
0         0.1422431503                      
1         0.1488859104                      
2         0.1225603133    

In [4]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 100].index

gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = 9999
print(gdf_parcels[class_column].value_counts())

label
110     23314
119      9525
111      5299
129      3660
112      3288
130      2990
108      1651
128      1372
114      1001
116       870
121       675
101       590
105       532
122       450
102       365
9999      304
106       250
104       153
Name: count, dtype: int64


In [5]:
display(gdf_parcels.head(3))

parcel_code  label             geometry  batch_number  eligible_pixel_count  \
0   266124977    110  POLYGON ((346567...             3                   44    
1    26857194    119  POLYGON ((336782...             3                   33    
2    26857195    119  POLYGON ((335690...             3                   89    

   meets_minimum_pixel_count  parcel_area_m2  approx_pixel_count  \
0                 True       4622.0831583770       46.2208315838   
1                 True       3296.4026005738       32.9640260057   
2                 True       8657.4707037060       86.5747070371   

   B02_bowley_skewness__20231001  B02_bowley_skewness__20231101  \
0        -0.2222223536                  -0.2606636878             
1         0.2777778640                   0.3008132642             
2         0.0789472523                   0.5600000745             

   B02_bowley_skewness__20231201  B02_bowley_skewness__20240101  \
0         0.3705885962                  -0.2437950298             
1         0.1538461355                   0.2293576163             
2         0.2142859519                   0.0392156147             

   B02_bowley_skewness__20240201  B02_bowley_skewness__20240301  \
0         0.0317449805                  -0.0531940864             
1         0.3023259239                  -0.1886796166             
2         0.2786888450                   0.6046514146             

   B02_bowley_skewness__20240401  B02_bowley_skewness__20240501  \
0         0.2727292431                   0.3005467017             
1         0.6962025615                   0.5952381903             
2         0.3614041600                   0.4193546837             

   B02_bowley_skewness__20240601  B02_bowley_skewness__20240701  \
0        -0.1412439525                   0.1864407636             
1         0.2212392881                  -0.3518521585             
2         0.0000000000                   0.2336454455             

   B02_bowley_skewness__20240801  B02_bowley_skewness__20240901  \
0         0.1384614679                  -0.0684901867             
1        -0.0140842262                  -0.2920352465             
2         0.1290320255                   0.2677169835             

   B02_coefficient_of_variation__20231001  \
0         0.1539387617                      
1         0.1977170996                      
2         0.2083519887                      

   B02_coefficient_of_variation__20231101  \
0         0.1340517814                      
1         0.1226033857                      
2         0.2390885263                      

   B02_coefficient_of_variation__20231201  \
0         0.1555584573                      
1         0.2268157149                      
2         0.2487263917                      

   B02_coefficient_of_variation__20240101  \
0         0.1321227188                      
1         0.2087659311                      
2         0.1694184639                      

   B02_coefficient_of_variation__20240201  \
0         0.0979855404                      
1         0.1757070871                      
2         0.1864479979                      

   B02_coefficient_of_variation__20240301  \
0         0.1214867041                      
1         0.3014574891                      
2         0.2711784960                      

   B02_coefficient_of_variation__20240401  \
0         0.1451981675                      
1         0.2926339985                      
2         0.2288398794                      

   B02_coefficient_of_variation__20240501  \
0         0.2022535334                      
1         0.2349692473                      
2         0.1295184390                      

   B02_coefficient_of_variation__20240601  \
0         0.1720276301                      
1         0.1197639817                      
2         0.1128968037                      

   B02_coefficient_of_variation__20240701  \
0         0.1422431503                      
1         0.1488859104                      
2         0.1225603133    

In [6]:
# NON_FEATURE_COLUMNS = {
#     parcel_id_column, class_column, "parcel_id", "parcel_code", "class", "label", "label_code", "initial_label", "initial_label_code", "period_start", "period_end",
#     "predicted_label", "predicted_label_code", "rfv_true", "svm_novelty_full", "parcel_area_m2", "eligible_pixel_count",
#     "parcel_id", "period_start", "period_end", "geometry", "batch_number", "prediction_probability",
#     "categ_code", "data_clean_mid", "has_rfv", "normal", "svm_outlier_full", "holdout_mid", "holdout_full", "meets_minimum_pixel_count"
# }

# BASE_COLUMNS = [parcel_id_column, class_column, "geometry"]
# remove_keywords = ["B02_", "B03_", "B04_", "count", "_mean", "_p25", "_p75", "_p10", "_p90", "_variance", "_coefficient_of_variation", "_bowley_skewness"] + list(NON_FEATURE_COLUMNS)

# keep_statistics_columns = [x for x in list(gdf.columns) if all(keyword not in x for keyword in remove_keywords) and x not in BASE_COLUMNS]
# keep_columns = BASE_COLUMNS + keep_statistics_columns
# df_final = gdf[keep_columns].copy()
# print(f"Columns dropped from the merged dataframe: {[x for x in list(gdf.columns) if x not in keep_columns]}")

# numeric_or_boolean = set(df_final.select_dtypes(include=["number", "bool"]).columns)
# features = [col for col in df_final.columns if col in numeric_or_boolean and col not in BASE_COLUMNS]

# print(f"Identified {len(features)} features for modeling: {features}")

In [7]:
gdf_parcels = gdf_parcels.to_crs("EPSG:2100")
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

extra_features = ["elevation_mean_m", "distance_to_nearest_lake_m", "GAIOIKANOT", "x", "y"]

# original_features = ["SAVI_median__20231201", "NDVI_median__20231101", "NDWI_median__20231201", "SAVI_median__20240101", "NDVI_median__20240701", "B04_median__20240701", "NDMI_median__20231101", "NDMI_median__20231201", "NDVI_median__20231001", "NDVI_median__20240901", "NDWI_median__20240801", "NDWI_median__20240101", "B08_median__20231201", "NDRE_median__20240801", "NDRE_median__20240201", "NDMI_median__20231001", "NDMI_median__20240101", "B04_median__20231001", "EVI_median__20240601", "B08_median__20240101", "B04_median__20240901", "B11_median__20240701", "NDMI_median__20240901", "NDMI_median__20240401", "B11_median__20240501", "B12_median__20240801", "B04_median__20231101", "EVI_sd__20240101", "EVI_sd__20231201", "NDMI_sd__20240701", "EVI_sd__20240701", "NDMI_sd__20240801", "NDMI_median__20240201", "EVI_sd__20240201", "NDMI_median__20240701", "EVI_median__20240301", "B11_median__20240401", "B08_median__20231001", "B08_median__20240401", "NDRE_sd__20240701", "B08_median__20240301", "B08_median__20240201", "B08_median__20240601", "B11_median__20240301", "NDWI_median__20240601", "B04_median__20240601", "EVI_sd__20240901", "B08_median__20240901", "B03_median__20231001", "NDWI_sd__20240801", "EVI_sd__20240601", "B08_sd__20231201", "NDMI_median__20240801", "B08_median__20240701", "B04_median__20240401", "B05_median__20240501", "NDMI_sd__20240201", "NDMI_median__20240501", "NDMI_sd__20240601", "B11_median__20240901", "B08_median__20240801", "NDVI_sd__20240101", "NDVI_sd__20240201", "B11_median__20231201", "NDMI_sd__20240901", "B08_sd__20240101", "B02_median__20231101", "B05_median__20240401", "NDWI_median__20240501", "B04_median__20240201", "B08_sd__20240301", "NDWI_sd__20240601", "NDMI_median__20240601", "B12_sd__20240701", "NDRE_sd__20240201", "B12_sd__20240801", "EVI_sd__20231101", "B12_median__20231101", "B08_sd__20240901", "B04_median__20231201", "B12_sd__20240601", "NDRE_sd__20240601", "B08_sd__20240701", "B08_sd__20240201", "B11_median__20231001", "NDRE_sd__20240901", "SAVI_sd__20240501", "NDMI_sd__20240101", "B08_sd__20240801", "B04_median__20240101", "EVI_sd__20240301", "B08_sd__20240501", "NDVI_sd__20231201", "B08_sd__20231101", "B08_median__20240501", "B05_median__20240301", "B05_median__20231201", "NDMI_sd__20240501", "B08_sd__20240601", "B03_median__20240201", "NDWI_sd__20240201"]
# original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in ["_mean", "_sd", "_min", "_max"])]

original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in ["_median", "_sd", "_range"]) and x not in extra_features and not x.startswith("B")]
base_cols = [parcel_id_column, class_column, "geometry"]
gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

Final features for modeling (221 features): ['EVI_median__20231001', 'EVI_median__20231101', 'EVI_median__20231201', 'EVI_median__20240101', 'EVI_median__20240201', 'EVI_median__20240301', 'EVI_median__20240401', 'EVI_median__20240501', 'EVI_median__20240601', 'EVI_median__20240701', 'EVI_median__20240801', 'EVI_median__20240901', 'EVI_range__20231001', 'EVI_range__20231101', 'EVI_range__20231201', 'EVI_range__20240101', 'EVI_range__20240201', 'EVI_range__20240301', 'EVI_range__20240401', 'EVI_range__20240501', 'EVI_range__20240601', 'EVI_range__20240701', 'EVI_range__20240801', 'EVI_range__20240901', 'EVI_sd__20231001', 'EVI_sd__20231101', 'EVI_sd__20231201', 'EVI_sd__20240101', 'EVI_sd__20240201', 'EVI_sd__20240301', 'EVI_sd__20240401', 'EVI_sd__20240501', 'EVI_sd__20240601', 'EVI_sd__20240701', 'EVI_sd__20240801', 'EVI_sd__20240901', 'NDMI_median__20231001', 'NDMI_median__20231101', 'NDMI_median__20231201', 'NDMI_median__20240101', 'NDMI_median__20240201', 'NDMI_median__20240301', '

,parcel_code,label,geometry,EVI_median__20231001,EVI_median__20231101,EVI_median__20231201,EVI_median__20240101,EVI_median__20240201,EVI_median__20240301,EVI_median__20240401,EVI_median__20240501,EVI_median__20240601,EVI_median__20240701,EVI_median__20240801,EVI_median__20240901,EVI_range__20231001,EVI_range__20231101,EVI_range__20231201,EVI_range__20240101,EVI_range__20240201,EVI_range__20240301,EVI_range__20240401,EVI_range__20240501,EVI_range__20240601,EVI_range__20240701,EVI_range__20240801,EVI_range__20240901,EVI_sd__20231001,EVI_sd__20231101,EVI_sd__20231201,EVI_sd__20240101,EVI_sd__20240201,EVI_sd__20240301,EVI_sd__20240401,EVI_sd__20240501,EVI_sd__20240601,EVI_sd__20240701,EVI_sd__20240801,EVI_sd__20240901,NDMI_median__20231001,NDMI_median__20231101,NDMI_median__20231201,NDMI_median__20240101,NDMI_median__20240201,NDMI_median__20240301,NDMI_median__20240401,NDMI_median__20240501,NDMI_median__20240601,NDMI_median__20240701,NDMI_median__20240801,NDMI_median__20240901,NDMI_range__20231001,NDMI_range__20231101,NDMI_range__20231201,NDMI_range__20240101,NDMI_range__20240201,NDMI_range__20240301,NDMI_range__20240401,NDMI_range__20240501,NDMI_range__20240601,NDMI_range__20240701,NDMI_range__20240801,NDMI_range__20240901,NDMI_sd__20231001,NDMI_sd__20231101,NDMI_sd__20231201,NDMI_sd__20240101,NDMI_sd__20240201,NDMI_sd__20240301,NDMI_sd__20240401,NDMI_sd__20240501,NDMI_sd__20240601,NDMI_sd__20240701,NDMI_sd__20240801,NDMI_sd__20240901,NDRE_median__20231001,NDRE_median__20231101,NDRE_median__20231201,NDRE_median__20240101,NDRE_median__20240201,NDRE_median__20240301,NDRE_median__20240401,NDRE_median__20240501,NDRE_median__20240601,NDRE_median__20240701,NDRE_median__20240801,NDRE_median__20240901,NDRE_range__20231001,NDRE_range__20231101,NDRE_range__20231201,NDRE_range__20240101,NDRE_range__20240201,NDRE_range__20240301,NDRE_range__20240401,NDRE_range__20240501,NDRE_range__20240601,NDRE_range__20240701,NDRE_range__20240801,NDRE_range__20240901,NDRE_sd__20231001,NDRE_sd__20231101,NDRE_sd__20231201,NDRE_sd__20240101,NDRE_sd__20240201,NDRE_sd__20240301,NDRE_sd__20240401,NDRE_sd__20240501,NDRE_sd__20240601,NDRE_sd__20240701,NDRE_sd__20240801,NDRE_sd__20240901,NDVI_median__20231001,NDVI_median__20231101,NDVI_median__20231201,NDVI_median__20240101,NDVI_median__20240201,NDVI_median__20240301,NDVI_median__20240401,NDVI_median__20240501,NDVI_median__20240601,NDVI_median__20240701,NDVI_median__20240801,NDVI_median__20240901,NDVI_range__20231001,NDVI_range__20231101,NDVI_range__20231201,NDVI_range__20240101,NDVI_range__20240201,NDVI_range__20240301,NDVI_range__20240401,NDVI_range__20240501,NDVI_range__20240601,NDVI_range__20240701,NDVI_range__20240801,NDVI_range__20240901,NDVI_sd__20231001,NDVI_sd__20231101,NDVI_sd__20231201,NDVI_sd__20240101,NDVI_sd__20240201,NDVI_sd__20240301,NDVI_sd__20240401,NDVI_sd__20240501,NDVI_sd__20240601,NDVI_sd__20240701,NDVI_sd__20240801,NDVI_sd__20240901,NDWI_median__20231001,NDWI_median__20231101,NDWI_median__20231201,NDWI_median__20240101,NDWI_median__20240201,NDWI_median__20240301,NDWI_median__20240401,NDWI_median__20240501,NDWI_median__20240601,NDWI_median__20240701,NDWI_median__20240801,NDWI_median__20240901,NDWI_range__20231001,NDWI_range__20231101,NDWI_range__20231201,NDWI_range__20240101,NDWI_range__20240201,NDWI_range__20240301,NDWI_range__20240401,NDWI_range__20240501,NDWI_range__20240601,NDWI_range__20240701,NDWI_range__20240801,NDWI_range__20240901,NDWI_sd__20231001,NDWI_sd__20231101,NDWI_sd__20231201,NDWI_sd__20240101,NDWI_sd__20240201,NDWI_sd__20240301,NDWI_sd__20240401,NDWI_sd__20240501,NDWI_sd__20240601,NDWI_sd__20240701,NDWI_sd__20240801,NDWI_sd__20240901,SAVI_median__20231001,SAVI_median__20231101,SAVI_median__20231201,SAVI_median__20240101,SAVI_median__20240201,SAVI_median__20240301,SAVI_median__20240401,SAVI_median__20240501,SAVI_median__20240601,SAVI_median__20240701,SAVI_median__20240801,SAVI_median__20240901,SAVI_range__20231001,SAVI_range__20231101,SAVI_range__20231201,SAVI_range__20240101,

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method="nearest",
    spatial_interpolation_max_distance_in_meters=500,

    cv_folds=5,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",  # or "by_group"

    random_state=SEED_NUMBER,
    n_jobs=4,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=3,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,
    interpretability_top_models=3,
    interpretability_include_shap=True,

    optimize_class_probabilities=False,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    label_balancing_method="random_oversample",  # alternatively: "none", "random_oversample" or "smote"
    selected_models=(
        #"gradient_boosting",
        #"hist_gradient_boosting",
        "bagging",
        "random_forest",
        # "decision_tree",
        "extra_trees",
        # "bayesian",
        #"xgboost",
        #"lightgbm",
        # "knn",
        "neural_network"
   ),
    max_map_geometries=20000,
    open_html_report=True,
)
pipeline = GeospatialClassificationPipeline(config)

In [9]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check


Function <run_check> started on 2026-08-16 18:46:30

Checking input data...

Resetting project output folder: C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1
Deleting folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\01_check'...
Successfully deleted folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\01_check' on attempt 1
Deleting folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\02_prepare'...
Successfully deleted folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\02_prepare' on attempt 1
Deleting folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\03_train'...
Successfully deleted folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\03_train' on attempt 1
Deleting folder 'C:\work_dir\kozani-2023-2024\3_ml_classification_tune_probabilities1\04_evaluate'...
Successfully deleted folder 'C:\work_dir\kozani-2023-2

{'rows_total': 55985,
 'rows_labeled': 55985,
 'rows_unknown_target': 0,
 'n_features_requested': 221,
 'n_features_active': 221,
 'numeric_features': ['EVI_median__20231001',
  'EVI_median__20231101',
  'EVI_median__20231201',
  'EVI_median__20240101',
  'EVI_median__20240201',
  'EVI_median__20240301',
  'EVI_median__20240401',
  'EVI_median__20240501',
  'EVI_median__20240601',
  'EVI_median__20240701',
  'EVI_median__20240801',
  'EVI_median__20240901',
  'EVI_range__20231001',
  'EVI_range__20231101',
  'EVI_range__20231201',
  'EVI_range__20240101',
  'EVI_range__20240201',
  'EVI_range__20240301',
  'EVI_range__20240401',
  'EVI_range__20240501',
  'EVI_range__20240601',
  'EVI_range__20240701',
  'EVI_range__20240801',
  'EVI_range__20240901',
  'EVI_sd__20231001',
  'EVI_sd__20231101',
  'EVI_sd__20231201',
  'EVI_sd__20240101',
  'EVI_sd__20240201',
  'EVI_sd__20240301',
  'EVI_sd__20240401',
  'EVI_sd__20240501',
  'EVI_sd__20240601',
  'EVI_sd__20240701',
  'EVI_sd__2024080

In [10]:
summary_prepare = pipeline.run_prepare()


Function <run_prepare> started on 2026-08-16 18:46:51

Preparing data...
Writing file: C:/work_dir/kozani-2023-2024/3_ml_classification_tune_probabilities1/02_prepare/label_mapping.csv


E:\Eleni\git_repo\satellites\src\ml_classification\ml_classification_pipeline\visuals\maps.py:156: UserWarning: Legend does not support handles for PatchCollection instances.
See: https://matplotlib.org/stable/tutorials/intermediate/legend_guide.html#implementing-a-custom-legend-handler
  ax.legend()



Prepared train/test split: 44788 train rows, 11197 test rows.

Execution time of function <run_prepare> ended on 2026-08-16 18:47:23 - duration: 00:00:32


In [ ]:
summary_train = pipeline.run_train()
summary_train


Function <run_train> started on 2026-08-16 18:47:23

Training models...
Reading file: C:/work_dir/kozani-2023-2024/3_ml_classification_tune_probabilities1/02_prepare/label_mapping.csv
____________________________________________________________________________________________________

Training model random_forest


In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict

In [ ]:
# Show only rows that were unknown during training
classified_gdf = pd.read_csv(os.path.join(str(OUTPUT_PATH), "05_predict", "final_predictions_preview.csv"))
classified_gdf[parcel_id_column] = classified_gdf[parcel_id_column].astype(str)
print(len(classified_gdf), "rows in classified_gdf")
display(classified_gdf.head())

In [ ]:
from ml_classification.ml_classification_pipeline.core.persistence import load_joblib
test_df = load_joblib(Path(f"{OUTPUT_PATH}/02_prepare/test_dataset.joblib"))
test_df[parcel_id_column] = test_df[parcel_id_column].astype(str)
print(len(test_df), "rows in test_df")
test_df.head()

In [ ]:
test_ids =list(test_df[parcel_id_column].unique())
classified_gdf = classified_gdf[classified_gdf[parcel_id_column].isin(test_ids)]
print(len(classified_gdf), "rows in classified_gdf after filtering with test_ids")
classified_gdf["accuracy"] = classified_gdf[class_column] == classified_gdf[f"{class_column}_prediction"]
accuracy = classified_gdf["accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")